# Práctica 1 · 01 — Descarga de datos de reanálisis

**Cambio Climático · Ingeniería Ambiental · ESPAM MFL** · Material CC-IAM-P1-01 · Versión 3.0 · octubre de 2026

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/ambiental/practica-1/P1_01_descarga_datos.ipynb)

**Objetivo.** Obtener, solo para el recuadro de su zona (provincia o cantón), las 9 variables de dos reanálisis: ERA5 (ECMWF) y CORe (NOAA).

**Producto.** La carpeta `Mi unidad/CC_datos/<zona>/` con los archivos NetCDF recortados y un inventario. Los demás cuadernos trabajan sobre ella.

**Requisito.** Su **propia** cuenta en el Climate Data Store, con la licencia de ERA5 aceptada, y su **propio** API Token (en Colab › Secrets como `CDSAPI_KEY`, o en la celda 1). No se comparten cuentas.

**Cómo trabajar.** En Colab: **Archivo › Guardar una copia en Drive**. Escriba solo en la celda **1** y ejecute las demás en orden.

<details><summary>Secuencia de la práctica y variables</summary>


| Orden | Cuaderno | Qué hace | Deja listo |
|---|---|---|---|
| 1 | `P1_01_descarga_datos` | Descarga ERA5 y CORe solo para el recuadro de su zona | `CC_datos/<zona>/` |
| 2 | `P1_02_series` | Recorta con el límite y promedia sobre la zona | Series diaria y mensual |
| 3 | `P1_03_climatologia` | Climatología 1995–2014 de las 8 variables con los dos reanálisis | Tablas y figuras (paso 1) |
| 4 | `P1_04_estacion_vs_reanalisis` | Contrasta una estación con ERA5 y CORe | Métricas y figuras (paso 2) |


| Variable | Columna | Unidad |
|---|---|---|
| Temperatura media, máxima y mínima a 2 m | `tmed`, `tmax`, `tmin` | °C |
| Precipitación | `pr` | mm/día · mm/mes |
| Humedad relativa a 2 m | `hr` | % |
| Humedad específica a 2 m | `q` | g/kg |
| Presión atmosférica en superficie | `ps` | hPa |
| Viento a 10 m: componentes, velocidad y dirección de procedencia | `u10`, `v10`, `ws`, `wd` | m/s · ° |

</details>

© 2026 Cristian Omar Paliz Acosta · ESPAM MFL · Todos los derechos reservados

## 1. Datos del estudiante

In [ ]:
ESTUDIANTE = "Nombre Apellido"
PROVINCIA  = "Manabí"   # la misma en todos los cuadernos
CANTON     = None       # None = toda la provincia; o el nombre de un cantón, p. ej. "Portoviejo"

ANIO_INI, ANIO_FIN = 1981, 2024   # periodo de datos del curso: no cambiar

# Credencial PERSONAL del Climate Data Store (ver sección 3). Déjelo vacío si la guardó en Secrets.
CDS_TOKEN = ""   # pegue aquí SU API Token, entre las comillas

## 2. Entorno de trabajo

Instala dos librerías, trae el código del curso desde GitHub (`herramientas/`), conecta su Drive y ubica la zona en la capa oficial de límites.

In [ ]:
!pip -q install cdsapi regionmask

import os, sys, pathlib, subprocess, datetime, platform, json
REPO_URL = "https://github.com/crispaliz/cambio-climatico-espam"
REPO_DIR = pathlib.Path(os.environ.get("CC_REPO", "/content/cambio-climatico-espam"))
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=False, stderr=subprocess.DEVNULL)
sys.path.insert(0, str(REPO_DIR))

from herramientas import limites, variables as V
from herramientas.reanalisis import era5, core, release
from herramientas.estaciones import calidad

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = pathlib.Path("/content/drive/MyDrive")
except Exception:
    BASE = pathlib.Path(".")

import numpy as np, pandas as pd, xarray as xr, matplotlib.pyplot as plt, geopandas as gpd
Z = limites.zona(PROVINCIA, CANTON)
GEOM, S = Z["geom"], Z["slug"]
DATOS = BASE / "CC_datos" / S            # datos descargados (cuaderno 1)
RES = BASE / "CC_Practica1" / S          # productos de la práctica (cuadernos 2 a 4)
for c in (DATOS, RES): c.mkdir(parents=True, exist_ok=True)
print(f"Zona: {Z['nombre']} ({Z['tipo']}, DPA {Z['codigo']}, región {Z['region']})")
print(f"Datos en {DATOS} · resultados en {RES}")

## 3. Su credencial personal del Climate Data Store

**Cada estudiante usa su propia cuenta y su propio API Token.** El CDS pone en cola y limita las solicitudes **por cuenta**: si todo el curso usara una sola, las descargas se bloquearían entre sí y la cuenta podría quedar suspendida.

Dónde está su token: entre a **cds.climate.copernicus.eu**, inicie sesión, abra su perfil (su nombre, arriba a la derecha) y copie el **API Token**.

Dos formas de usarlo (elija una):

| Forma | Cómo | Recomendación |
|---|---|---|
| **A. Secrets de Colab** | Ícono de llave a la izquierda › *Add new secret* › nombre `CDSAPI_KEY`, valor: su token › active el acceso para este cuaderno | La más segura: el token no queda escrito en el cuaderno |
| **B. En la celda 1** | Pegue el token entre las comillas de `CDS_TOKEN` | Práctica, pero **no comparta** su copia del cuaderno con el token escrito |

Si no hace ninguna de las dos, la celda siguiente se lo pedirá y no quedará escrito en ningún lado. La celda muestra solo el inicio y el final del token, para que confirme que es el suyo.

In [ ]:
print("Credencial activa:", era5.credenciales(CDS_TOKEN))

## 4. Qué se descarga y cómo se cuida el servidor

| Fuente | Malla | Frecuencia | De dónde | Solicitudes | Credencial |
|---|---|---|---|---|---|
| ERA5 mensual | 0,25° (~28 km) | Mensual | Climate Data Store | **1** con todas las variables | Su API Token personal |
| ERA5 terreno | 0,25° | Fijo | Climate Data Store | 1 | Su API Token personal |
| ERA5 diario | 0,25° | Diaria (día local) | Release `era5-v1` del curso, archivo de su provincia | 1 | Ninguna |
| CORe diario y mensual | ~1,9° | Diaria (01:00–01:00 local) y mensual | Release `core-v1` del curso, archivo de su provincia | 2 | Ninguna |

- Los servidores recortan por **rectángulo**: se pide el rectángulo mínimo que contiene la zona, ajustado a la malla de cada fuente. El recorte exacto con el límite se hace en el cuaderno 2.
- Si un archivo ya está en su Drive, **no se vuelve a pedir**. Si una solicitud falla, se reintenta con espera creciente.
- De los releases se guarda **solo el recorte de su zona**; el archivo provincial se borra al terminar.

In [ ]:
AREA = limites.recuadro(GEOM, resolucion=0.25, margen_celdas=1)       # ERA5
AREA_CORE = limites.recuadro(GEOM, resolucion=1.875, margen_celdas=1)  # CORe: celdas más grandes
print("Recuadro ERA5 [N, O, S, E]:", AREA)
print("Recuadro CORe [N, O, S, E]:", AREA_CORE)

ax = limites.cargar("provincias").boundary.plot(color="0.75", linewidth=0.5, figsize=(5, 6))
gpd.GeoSeries([GEOM]).plot(ax=ax, color="#F2A65A")
for (n, o, s, e), estilo, et in ((AREA, "k--", "ERA5"), (AREA_CORE, "b:", "CORe")):
    ax.plot([o, e, e, o, o], [n, n, s, s, n], estilo, label=et)
ax.set_xlim(AREA_CORE[1] - 0.5, AREA_CORE[3] + 0.5); ax.set_ylim(AREA_CORE[2] - 0.5, AREA_CORE[0] + 0.5)
ax.legend(); ax.set_title(f"{Z['nombre']}: recuadros de descarga"); plt.show()

## 5. ERA5 mensual desde el Climate Data Store

**Una sola solicitud** con temperatura y punto de rocío a 2 m, precipitación, presión en superficie y a nivel del mar, y viento a 10 m (u, v). Con el punto de rocío se calculan la humedad relativa y, junto con la presión, la humedad específica. Además, una solicitud pequeña con la **geopotencial**, que da la altura del terreno del modelo (se usa en el cuaderno 4). La solicitud entra en cola y puede tardar varios minutos.

In [ ]:
anios = range(ANIO_INI, ANIO_FIN + 1)
ARCH = {}
ARCH["era5_mensual"] = era5.descargar_mensual(anios, AREA, DATOS / f"era5_mensual_{S}_{ANIO_INI}_{ANIO_FIN}.nc")
ARCH["era5_terreno"] = era5.descargar_terreno(AREA, DATOS / f"era5_terreno_{S}.nc")
print("ERA5 mensual y terreno listos")

## 6. ERA5 diario y CORe, preparados por el docente

Se trae el archivo de su provincia, se recorta al recuadro de su zona y al periodo, y se borra el archivo provincial.

In [ ]:
ARCH["era5_diario"] = release.obtener("era5", "diario", Z, AREA, anios, DATOS)
ARCH["core_diario"] = release.obtener("core", "diario", Z, AREA_CORE, anios, DATOS)
ARCH["core_mensual"] = release.obtener("core", "mensual", Z, AREA_CORE, anios, DATOS)
for k, v in ARCH.items():
    print(f"{k:14s}", "falta (release aún no publicado)" if v is None else v.name)

## 7. Inventario y control

Periodo, número de pasos de tiempo, malla, variables y porcentaje de vacíos de cada archivo. El inventario queda junto a los datos: es la evidencia de qué se descargó, de dónde y cuándo.

In [ ]:
filas = []
for k, f in ARCH.items():
    if f is None:
        filas.append({"archivo": k, "estado": "no disponible"}); continue
    ds = era5.abrir_ds(f) if k.startswith("era5_m") or k == "era5_terreno" else xr.open_dataset(f)
    t = ds["time"].values if "time" in ds.dims else [np.datetime64("NaT")]
    filas.append({"archivo": f.name, "estado": "ok", "variables": ", ".join(ds.data_vars),
                  "desde": str(np.min(t))[:10], "hasta": str(np.max(t))[:10], "pasos": int(ds.sizes.get("time", 1)),
                  "malla": f"{ds.sizes['lat']}×{ds.sizes['lon']}", "MB": round(f.stat().st_size / 1e6, 2),
                  "vacíos_%": round(100 * float(ds[list(ds.data_vars)[0]].isnull().mean()), 2)})
INVENTARIO = pd.DataFrame(filas)
display(INVENTARIO)
print(f"Espacio total en su Drive: {sum(f.stat().st_size for f in ARCH.values() if f) / 1e6:.1f} MB")

meta = {"estudiante": ESTUDIANTE, "zona": {k: v for k, v in Z.items() if k != "geom"}, "periodo": [ANIO_INI, ANIO_FIN],
        "recuadro_era5": AREA, "recuadro_core": AREA_CORE, "fecha": datetime.date.today().isoformat(),
        "fuentes": {"era5_mensual": f"{era5.MENSUAL} (DOI {era5.DOI[era5.MENSUAL]})",
                    "era5_diario": f"release {release.ETIQUETAS['era5']}", "core": f"release {release.ETIQUETAS['core']}"},
        "archivos": {k: (None if v is None else v.name) for k, v in ARCH.items()}}
(DATOS / "inventario.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
INVENTARIO.to_csv(DATOS / "inventario.csv", index=False, encoding="utf-8")

In [ ]:
# Vista rápida: media del periodo sobre el recuadro, con el límite de la zona
M = era5.mensual_estandar(ARCH["era5_mensual"])
vistas = [("ERA5 · T media (°C)", M["tmed"].mean("time")), ("ERA5 · HR (%)", M["hr"].mean("time"))]
if ARCH.get("core_mensual"):
    vistas.append(("CORe · T media (°C)", xr.open_dataset(ARCH["core_mensual"])["tmed"].mean("time")))
fig, ax = plt.subplots(1, len(vistas), figsize=(4.5 * len(vistas), 4))
for a, (t, da) in zip(np.atleast_1d(ax), vistas):
    da.plot(ax=a, cmap="RdYlBu_r" if "T" in t else "YlGnBu"); gpd.GeoSeries([GEOM]).boundary.plot(ax=a, color="k", lw=0.8); a.set_title(t)
plt.tight_layout(); plt.show()

## 8. Antes de seguir

- ¿Todos los archivos tienen estado `ok`? Si falta uno de los releases, avise al docente; puede seguir con lo que tenga.
- ¿Cuántas celdas de cada reanálisis cubren su zona? Esa diferencia de resolución pesará en todo lo que sigue.

**Siguiente:** `P1_02_series`.

**Citas.** Hersbach, H. et al. (2020). The ERA5 global reanalysis. *Q. J. R. Meteorol. Soc.*, 146, 1999–2049 · Hersbach, H. et al. (2023). *ERA5 monthly averaged data on single levels from 1940 to present*. C3S CDS. DOI 10.24381/cds.f17050d7 · NOAA/NCEP Climate Prediction Center. CORe, Conventional Observation Reanalysis.